In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# colab-only
!pip install --pre giskard-scan "giskard-agents[openai]" openai

`quality_scan` generates and runs a suite that probes whether an agent is
actually correct: does it stay grounded in your documents, does it fold under a
confidently wrong premise, does it invent answers for things your knowledge base
never covered.

It lives in the `giskard-scan` package, next to `vulnerability_scan`.

```bash
pip install --pre giskard-scan "giskard-agents[openai]"
```

## 1. Define a target and a knowledge base

`quality_scan` is async, and it needs a knowledge base: the documents the agent
is supposed to be grounded in. The target below is a deliberately naive agent
that answers from the model's own priors instead of retrieving anything.

In [3]:
from openai import AzureOpenAI

from giskard.checks import set_default_generator
from giskard.agents.generators import Generator

set_default_generator(Generator(model="azure_ai/gpt-4.1-nano"))

client = AzureOpenAI(
    api_key=os.environ["AZURE_AI_API_KEY"],
    azure_endpoint=os.environ["AZURE_AI_ENDPOINT"],
    api_version="2024-10-21",
)


def support_agent(inputs: str) -> str:
    """A naive support agent: no retrieval, it just answers from the model.

    The parameter must be named `inputs` — that is the name the scan injects.
    """
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a support agent for a SaaS billing product. Answer briefly."},
            {"role": "user", "content": inputs},
        ],
    )
    return response.choices[0].message.content

The knowledge base takes a plain list of strings, or `KnowledgeBase.from_texts`,
or `Document` objects when you want tags carried alongside the content.

In [4]:
from giskard.scan import Document, KnowledgeBase

kb = KnowledgeBase(
    documents=(
        Document(content="Refunds are available within 30 days of purchase.", tags=["billing"]),
        Document(content="Pro plan costs $49 per seat per month, billed annually.", tags=["billing"]),
        Document(content="Support is available Monday to Friday, 9am-6pm CET.", tags=["support"]),
        Document(content="We do not ship hardware; all products are software-only.", tags=["product"]),
        Document(content="Data is stored in the EU (Frankfurt) region.", tags=["product"]),
    )
)

print("documents:", len(kb.documents))

documents: 5


A bare `str` is rejected with a `TypeError` — otherwise `from_texts` would
happily build one document per character. Embeddings are computed lazily, in one
batch, the first time a generator needs nearest-neighbour retrieval, so creating
a `KnowledgeBase` costs nothing.

### Omitting the knowledge base

Every quality generator is knowledge-base driven. Pass no `knowledge_base` — or
an empty one — and the scan warns and skips all of them:

```text
RuntimeWarning: quality_scan received no knowledge base;
knowledge-base quality scenarios will be skipped.
```

The scan still runs, but there is nothing to generate, so the report comes back
empty. Worth treating that warning as an error in your own tooling.

## 2. Run the scan

`max_scenarios` is a total budget across *all* generators, divided between them.
Keep it small while you iterate — this page uses 4 to stay cheap.

In [5]:
import asyncio

from giskard.scan import quality_scan

result = asyncio.run(
    quality_scan(
        target=support_agent,
        description="A customer-support agent for a SaaS billing product.",
        languages=["en"],
        knowledge_base=kb,
        max_scenarios=4,
        target_mode="singleturn",
    )
)

Task was destroyed but it is pending!
task: <Task pending name='Task-86' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-87' 
coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/rich/text.py:144: RuntimeWarning: coroutine 'Kernel.shell_main' was never 
awaited
  def __init__(
RuntimeWarning: Enable tracemalloc to get the object allocation traceback
Task was destroyed but it is pending!
task: <Task pending name='Task-87' coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-88' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-89' 
coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-89' coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-90' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-91' 
coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-91' coro=<Kernel.shell_main() running at 
/private/tmp/claude-501/-Users-davidberenstein-Programming-giskard-docs/2a06057b-9559-4f21-a767-83d5e14fba85/scratc
hpad/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-92' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
/private/tmp/claude-501/-Users-d

`quality_scan` prints the grouped report itself and returns a `SuiteResult`.

## 3. Read the report

### `group_by` defaults to `"component"`

A quality scan groups its report by `component:` tags — `llm`, `retrieval`,
`history` — because what you want to know is which part of the pipeline is
broken: the model, the retriever, or conversation-history handling.
`vulnerability_scan` defaults to `group_by="threat-type"` instead, where the
interesting question is what kind of attack got through.

Both accept any annotation key, and `group_by=None` prints the ungrouped report.
You can also regroup after the fact, without re-running anything:

In [6]:
regrouped = result.group_by("quality")

for name, stats in regrouped.groups.items():
    print(f"{name}: {stats.passed} passed / {stats.failed} failed (pass rate {stats.pass_rate})")

direct-hallucination: 1 passed / 0 failed (pass rate 1.0)
sycophancy-hallucinations: 0 passed / 1 failed (pass rate 0.0)


### The `recommendation` field

A `SuiteResult` returned by `quality_scan` carries a `recommendation`: an
LLM-generated prose summary of what failed and what to do about it, built from
the per-`component` and per-`quality` pass rates.

In [7]:
if result.recommendation:
    print(result.recommendation)
else:
    print("No recommendation: nothing failed.")

- Improve the response accuracy of the `llm` component to ensure that it avoids agreeing with false claims, particularly in scenarios where user bias could lead the agent astray. This will strengthen the agent's ability to handle sycophancy-hallucinations effectively.
- Implement better detection mechanisms for user bias, allowing the agent to maintain factual integrity even in conversations that may prompt agreement with incorrect information.


Three caveats. It is quality-only — `vulnerability_scan` does not produce one.
It is empty when nothing failed (`result.failures_and_errors` is empty). And it
is best-effort: generating it costs an extra LLM call, and if that call fails
the exception is logged and `recommendation` falls back to `""` rather than
taking the scan result down with it. So guard with `if result.recommendation:`
instead of assuming a string is there.

## What each generator probes

`quality_scan` runs the whole quality registry:

| Generator | What it probes |
| --- | --- |
| `HallucinationScenarioGenerator` | Answers that contradict the retrieved documents. Tagged `quality:direct-hallucination`, `component:llm`. |
| `SycophancyScenarioGenerator` | Whether the agent caves when the user asserts a plausible premise the documents contradict. Tagged `quality:sycophancy-hallucinations`, `component:llm`. |
| `SplitQuestionsScenarioGenerator` | Context in message one, the question in message two — does the agent carry the context? Tagged `quality:split-questions`, `component:history`. |
| `MultiTopicScenarioGenerator` | Multi-turn questions that hop across knowledge-base topics. Tagged `quality:multi-topic-questions`, `component:retrieval`, `component:history`. |
| `OutOfScopeScenarioGenerator` | Precise, plausible-sounding things your documents never mention — does the agent fabricate? Tagged `quality:fabricated-hallucination`, `component:llm`, `component:retrieval`. |

### Narrowing to one family

To probe a single behaviour, skip `quality_scan` and build the suite yourself
with `generate_suite`:

```python
from giskard.scan import SycophancyScenarioGenerator, generate_suite

suite = asyncio.run(
    generate_suite(
        description="A customer-support agent for a SaaS billing product.",
        languages=["en"],
        generators=[SycophancyScenarioGenerator()],
        knowledge_base=kb,
        max_scenarios=2,
    )
)

result = asyncio.run(suite.run(support_agent, parallel=True))
```

Watch the `parallel` default when you do. `quality_scan` and
`vulnerability_scan` both default to `parallel=True`, but `Suite.run` defaults
to `parallel=False`. So the moment you drop down to `generate_suite` +
`suite.run` — the snippet above, or re-running a persisted suite — execution
quietly becomes serial and gets much slower. Pass `parallel=True` explicitly,
and cap it with `max_concurrency` if your provider rate-limits you.
`max_concurrency` only affects scheduling when `parallel=True`, though invalid
values are rejected either way. Scenario *generation* is always concurrent;
`parallel` only controls suite *execution*.

## Full signature

```python
async def quality_scan(
    target,
    description: str,
    languages: list[str],
    *,
    knowledge_base: KnowledgeBase | list[str] | None = None,
    max_scenarios: int | None = None,
    seed: int = 42,
    group_by: str | None = "component",
    parallel: bool = True,
    max_concurrency: int | None = None,
    return_exception: bool = False,
    target_mode: TargetMode = DEFAULT_TARGET_MODE,
) -> SuiteResult: ...
```

- `max_scenarios` — total budget across *all* generators, divided between them.
  `None` lets each generator use its own default. Set it explicitly to keep
  cost predictable.
- `seed` — defaults to `42`; the same arguments and seed produce the same
  scenarios.
- `return_exception` — `False` (default) lets an input-generation failure abort
  the scan. `True` records it as an errored result and keeps going. Use `True`
  for unattended runs, `False` while you are still debugging your target.
- `target_mode` — `"singleturn"` skips generators that are multi-turn by design
  and caps turn budgets to 1 on the rest. Pass it for any target that does not
  accept conversation history, otherwise you will be scored on turns your agent
  never saw.

## Quality scan vs vulnerability scan

| | `quality_scan` | `vulnerability_scan` |
| --- | --- | --- |
| Asks | Is the agent correct and grounded? | Is the agent safe? |
| Needs | A `knowledge_base` | Nothing beyond a description |
| Default `group_by` | `"component"` | `"threat-type"` |
| `recommendation` | Yes | No |
| Extra options | — | `commercial_use` (filters datasets) |

You want both: a scan-clean agent can still be confidently wrong, and a
perfectly grounded agent can still be jailbroken. See
[Scan for vulnerabilities](/oss/solutions/scan-vulnerabilities) for the safety
half.